# 🔍 Visualización de Anomalías en Runs Nuevas (Evaluación)

Este módulo interactivo permite analizar los resultados del modelo aplicado a datos no vistos (evaluación), explorando:

- Señal `CCL` suavizada.
- Score de anomalía (Isolation Forest) por tramos.
- Tramos más anómalos resaltados.
- Exportación profesional de resultados.


### 💾 Celda 2 – Carga del dataset de evaluación

In [149]:
import pandas as pd

# Cargar resultados
df_eval = pd.read_csv(r"C:\Developer\fundamentos\data\ccl_eval_scores.csv")

# Asegurar orden correcto
df_eval = df_eval.sort_values(by=["pozo", "etapa", "DEPT"]).reset_index(drop=True)

# Verificación
print(f"Total de filas: {len(df_eval)}")
df_eval.head()


Total de filas: 13760


,DEPT,CCL,TENS,archivo_origen,pozo,sentido,etapa,CCL_norm,dCCL,abs_dCCL,...,CCL_norm_max,CCL_norm_min,abs_dCCL_mean,abs_dCCL_std,abs_dCCL_max,TENS_mean,TENS_std,TENS_max,score_iso,anomaly_iso
0,2799.8928,0.00557,802.00006,BPO-2702_E19_Down__01Feb25_154156.las,BPO-2702,Down,E19,1.116232,NaN,NaN,...,10.0,-9.763527,1.548248,1.445324,19.162325,828.512329,66.340158,1381.99994,-0.051423,1
1,2800.0452,0.00128,802.00006,BPO-2702_E19_Down__01Feb25_154156.las,BPO-2702,Down,E19,0.256513,-0.859719,0.859719,...,10.0,-9.763527,1.548248,1.445324,19.162325,828.512329,66.340158,1381.99994,-0.085707,1
2,2800.1976,-0.00070,794.99996,BPO-2702_E19_Down__01Feb25_154156.las,BPO-2702,Down,E19,-0.140281,-0.396794,0.396794,...,10.0,-9.763527,1.548248,1.445324,19.162325,828.512329,66.340158,1381.99994,-0.101926,1
3,2800.3500,-0.00214,794.99996,BPO-2702_E19_Down__01Feb25_154156.las,BPO-2702,Down,E19,-0.428858,-0.288577,0.288577,...,10.0,-9.763527,1.548248,1.445324,19.162325,828.512329,66.340158,1381.99994,-0.107611,1
4,2800.5024,0.00815,794.99996,BPO-2702_E19_Down__01Feb25_154156.las,BPO-2702,Down,E19,1.633267,2.062124,2.062124,...,10.0,-9.763527,1.548248,1.445324,19.162325,828.512329,66.340158,1381.99994,-0.088696,1


### 🧰 Celda 3 – Herramientas interactivas

In [150]:
import plotly.graph_objs as go
import plotly.colors as colors
import ipywidgets as widgets
from IPython.display import display

# Dropdowns dinámicos
pozo_dropdown = widgets.Dropdown(options=sorted(df_eval["pozo"].unique()), description="Pozo:")
etapa_dropdown = widgets.Dropdown(description="Etapa:")

def update_etapas(pozo_sel):
    etapas = df_eval[df_eval["pozo"] == pozo_sel]["etapa"].unique()
    etapa_dropdown.options = sorted(etapas)

pozo_dropdown.observe(lambda change: update_etapas(change["new"]), names="value")
update_etapas(pozo_dropdown.value)

step_dropdown = widgets.Dropdown(options=[2.5, 5, 10, 15, 25, 50], value=50, description="Paso (m):")

display(pozo_dropdown, etapa_dropdown, step_dropdown)


Dropdown(description='Pozo:', options=('BPO-2702',), value='BPO-2702')

Dropdown(description='Etapa:', options=('E19',), value=None)

Dropdown(description='Paso (m):', index=5, options=(2.5, 5, 10, 15, 25, 50), value=50)

### 📊 Celda 4 – Función de visualización completa

In [157]:
def plot_eval_riesgo(df, pozo, etapa, step):
    df_et = df[(df["pozo"] == pozo) & (df["etapa"] == etapa)].sort_values("DEPT").copy()
    df_et["CCL_smooth"] = df_et["CCL"].rolling(window=25, min_periods=1).mean()
    bin_col = f"DEPT_bin_{step}"
    df_et[bin_col] = (df_et["DEPT"] // step) * step

    # Score promedio por tramo
    grouped = df_et.groupby(bin_col)["score_iso"].mean().reset_index().rename(columns={"score_iso": f"score_mean_{step}"})
    score_col = f"score_mean_{step}"
    top5 = grouped.sort_values(score_col, ascending=False).head(5).reset_index(drop=True)

    # Colores según severidad
    max_s = top5[score_col].max()
    min_s = top5[score_col].min()
    scale = colors.sequential.OrRd

    def s2color(score):
        idx = int(((score - min_s) / (max_s - min_s + 1e-5)) * (len(scale) - 1))
        return scale[idx]

    # Gráfico
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_et["CCL_smooth"],
        y=df_et["DEPT"],
        mode='lines',
        name='CCL suavizado',
        line=dict(color='blue')
    ))

    fig.add_trace(go.Scatter(
        x=grouped[score_col],
        y=grouped[bin_col],
        mode='lines+markers',
        name=f'Score medio cada {step}m',
        line=dict(color='black', width=2),
        marker=dict(size=6)
    ))

    for _, row in top5.iterrows():
        y0 = row[bin_col]
        y1 = y0 + step
        fig.add_shape(
            type="rect", x0=0, x1=1, xref="paper",
            y0=y0, y1=y1, yref="y",
            fillcolor=s2color(row[score_col]), opacity=0.3,
            line_width=0, layer="below"
        )

    fig.update_layout(
        title=f"Pozo: {pozo} | Etapa: {etapa} | Paso: {step} m",
        xaxis_title="Valor",
        yaxis_title="Profundidad (DEPT)",
        yaxis_autorange="reversed",
        height=700,
        margin=dict(l=20, r=20, t=50, b=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        font=dict(family="Arial", size=14)
    )

    fig.show()

    print(f"📋 Top 5 tramos más anómalos ({step} m):")
    display(top5)

    return fig, top5


### 💾 Celda 5 – Ejecutar visualización y exportar informe

In [159]:
fig, top5 = plot_eval_riesgo(df_eval, pozo_dropdown.value, etapa_dropdown.value, step_dropdown.value)

# Exportar si querés
#nombre_archivo = f"reporte_eval_{pozo_dropdown.value}_etapa{etapa_dropdown.value}_{step_dropdown.value}m.html"
#fig.write_html(nombre_archivo)
#print(f"✅ Gráfico exportado a: {nombre_archivo}")

# Guardar tabla top5
#top5.to_csv(nombre_archivo.replace(".html", "_top5.csv"), index=False)


📋 Top 5 tramos más anómalos (15 m):


,DEPT_bin_15,score_mean_15
0,2790.0,-0.098844
1,2850.0,-0.101084
2,3180.0,-0.104878
3,4785.0,-0.105198
4,3075.0,-0.106627
